In [74]:
# from gulps.synthesis_plugin import GulpsSynthesisPlugin
import numpy as np
from monodromy.render import _plot_coverage_set
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.circuit.library import (
    CXGate,
    RZXGate,
    UGate,
    UnitaryGate,
    XXPlusYYGate,
    iSwapGate,
)
from qiskit.circuit.random import random_circuit
from qiskit.quantum_info import Operator, average_gate_fidelity
from qiskit.quantum_info.random import random_unitary
from qiskit.transpiler import (
    InstructionProperties,
    PassManager,
    Target,
    generate_preset_pass_manager,
)
from qiskit.transpiler.passes import Optimize1qGatesDecomposition
from tqdm import tqdm
from weylchamber import c1c2c3

from gulps.gulps_synthesis import GulpsDecomposer
from gulps.synthesis_pass import GulpsDecompositionPass
from gulps.utils.invariants import GateInvariants
from gulps.utils.logging_config import logger

# logger.setLevel("DEBUG")
logger.setLevel("INFO")


### Usage as a Decomposer

In [75]:
gate_set = [
    CXGate(),
    CXGate().power(1 / 2),
    iSwapGate().power(1 / 2),
    iSwapGate().power(1 / 3),
]
costs = [1.0, 1 / 2, 1 / 2, 1 / 3]
decomposer = GulpsDecomposer(gate_set, costs, precompute_polytopes=True)

In [ ]:
N = 2_000
fidelities = []

for idx in tqdm(range(N)):
    u = random_unitary(4, seed=idx)
    v = Operator(decomposer(u))
    fid = average_gate_fidelity(u, v)
    fidelities.append(fid)

    if fid < 1 - 1e-6:
        print(f"Unitary {idx} fidelity is low: {fid:.8f}")
        print("Canonical invariants:")
        print("U:", c1c2c3(u))
        print("V:", c1c2c3(v))
        print("\n")
        continue

# Summary statistics
fidelities = np.array(fidelities)
print(f"\nSummary across {len(fidelities)} samples:")
print(f"  Median fidelity: {np.median(fidelities)}")
print(f"  Mean fidelity:   {np.mean(fidelities)}")
print(f"  Minimum fidelity:{np.min(fidelities)}")

  0%|          | 0/2000 [00:00<?, ?it/s]

 88%|████████▊ | 1753/2000 [00:51<00:06, 35.29it/s]

Unitary 1737 fidelity is low: 0.99999759
Canonical invariants:
U: (np.float64(0.463494), np.float64(0.37573474), np.float64(0.13693979))
V: (np.float64(0.46406276), np.float64(0.37484081), np.float64(0.13725073))




100%|██████████| 2000/2000 [01:00<00:00, 33.11it/s]


Summary across 2000 samples:
  Median fidelity: 1.0
  Mean fidelity:   0.999999998710135
  Minimum fidelity:0.9999975932424071


In [ ]:
# # %reload_ext  snakeviz
# %load_ext snakeviz
# import cProfile

# u = random_unitary(4, seed=0)
# cProfile.run("decomposer._run(u)", "profile_timings/temph.prof")